In [13]:
# Core libraries
!pip install -U pandas numpy scikit-learn

# Hugging‑Face 🤗 datasets
!pip install -U datasets

# Gradient‑boosting (choose one or both)
!pip install -U xgboost lightgbm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path
import logging
from datetime import datetime

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, QuantileTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score, f1_score
import xgboost as xgb
import lightgbm as lgb

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data
from datasets import load_dataset

# Persistence
import pickle

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)

In [15]:

# ----------------------------------------------------------------------
# 1. Load raw data ------------------------------------------------------
# ----------------------------------------------------------------------
def load_raw() -> pd.DataFrame:
    """
    Load the Hugging‑Face dataset and return a pandas DataFrame.
    """
    ds = load_dataset("mahmoudalyosify/SCRAP", split="train")
    df = pd.DataFrame(ds)
    # Ensure numeric dtypes (some columns may be read as objects)
    numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")
    return df


In [ ]:
data=load_raw()
data.head()

Repo card metadata block was not found. Setting CardData to empty.


,event_id,time_to_tca,mission_id,risk,max_risk_estimate,max_risk_scaling,miss_distance,relative_speed,relative_position_r,relative_position_t,...,t_sigma_rdot,c_sigma_rdot,t_sigma_tdot,c_sigma_tdot,t_sigma_ndot,c_sigma_ndot,F10,F3M,SSN,AP
0,0,1.566798,5,-10.204955,-7.834756,8.602101,14923.0,13792.0,453.8,5976.6,...,0.147350,58.272095,0.004092,0.165044,0.002987,0.386462,89.0,83.0,42.0,11.0
1,0,1.207494,5,-10.355758,-7.848937,8.956374,14544.0,13792.0,474.3,5821.2,...,0.059672,57.966413,0.003753,0.164383,0.002933,0.386393,89.0,83.0,42.0,11.0
2,0,0.952193,5,-10.345631,-7.847406,8.932195,14475.0,13792.0,474.6,5796.2,...,0.039258,57.907599,0.003576,0.164352,0.002967,0.386381,89.0,83.0,42.0,11.0
3,0,0.579669,5,-10.337809,-7.845880,8.913444,14579.0,13792.0,472.7,5838.9,...,0.022066,57.993905,0.003298,0.164309,0.002918,0.386400,89.0,83.0,40.0,14.0
4,0,0.257806,5,-10.391260,-7.852942,9.036838,14510.0,13792.0,478.7,5811.1,...,0.015075,57.946717,0.003670,0.164172,0.003220,0.386388,89.0,83.0,40.0,14.0


In [16]:
# ----------------------------------------------------------------------
# 2. Filter out CDMs that would not be available at prediction time -----
# ----------------------------------------------------------------------
def filter_by_time(df: pd.DataFrame, min_days: float = 2.0) -> pd.DataFrame:
    """
    Keep only rows with time_to_tca >= min_days.
    """
    return df[df["time_to_tca"] >= min_days].copy()


In [17]:

# ----------------------------------------------------------------------
# 3. Helper: retrieve the last risk value per event (after filtering) --
# ----------------------------------------------------------------------
def _last_risk(series: pd.Series) -> float:
    """Series is already sorted chronologically → last element = final risk."""
    return series.iloc[-1]


In [18]:
# ==============================================================================
# IMPROVED: 4. Advanced Time-Series Aggregation Per Event
# ==============================================================================
def aggregate_event_advanced(df: pd.DataFrame) -> pd.DataFrame:
    """
    IMPROVED aggregation: Beyond mean/std/min/max, extract rich time-series features.
    Includes: trends, percentiles, volatility, rate of change, etc.
    """
    target_col = "risk"
    numeric_cols = [
        "relative_position_r", "relative_position_t", "relative_position_n",
        "relative_velocity_r", "relative_velocity_t", "relative_velocity_n",
        "miss_distance", "relative_speed",
        "t_position_covariance_det", "c_position_covariance_det",
        "t_sigma_r", "c_sigma_r", "t_sigma_t", "c_sigma_t",
        "t_sigma_n", "c_sigma_n",
        "t_j2k_sma", "c_j2k_sma",
        "t_j2k_ecc", "c_j2k_ecc",
        "t_j2k_inc", "c_j2k_inc",
        "F10", "F3M", "SSN", "AP",
    ]

    df = df.sort_values(["event_id", "time_to_tca"], ascending=[True, True])

    # Function to compute advanced statistics
    def compute_advanced_stats(series):
        if len(series) < 2:
            series = pd.Series([series.iloc[0]] if len(series) == 1 else [0])
        
        stats = {
            'mean': series.mean(),
            'std': series.std(),
            'min': series.min(),
            'max': series.max(),
            'p25': series.quantile(0.25),
            'p50': series.quantile(0.50),
            'p75': series.quantile(0.75),
            'range': series.max() - series.min(),  # Max - Min
            'cv': series.std() / (series.mean() + 1e-9),  # Coefficient of variation
            'skew': series.skew(),  # Skewness
            'trend': (series.iloc[-1] - series.iloc[0]) if len(series) > 1 else 0,  # Last - First
            'slope': np.polyfit(np.arange(len(series)), series, 1)[0] if len(series) > 1 else 0,  # Linear trend
        }
        return stats
    
    # Build aggregation dict with advanced stats
    agg_dict = {}
    for col in numeric_cols:
        agg_dict[col] = compute_advanced_stats if callable(compute_advanced_stats) else ["mean", "std", "min", "max"]
    
    # Use groupby apply for custom functions
    event_agg_list = []
    for event_id, group in df.groupby("event_id"):
        row_dict = {"event_id": event_id}
        
        # Add risk (final value)
        row_dict["risk"] = group[target_col].iloc[-1]
        
        # Add advanced stats for each numeric column
        for col in numeric_cols:
            if col in group.columns:
                series = group[col].dropna()
                if len(series) > 0:
                    stats = compute_advanced_stats(series)
                    for stat_name, stat_value in stats.items():
                        row_dict[f"{col}_{stat_name}"] = stat_value
        
        event_agg_list.append(row_dict)
    
    event_agg = pd.DataFrame(event_agg_list)
    logger.info(f"Advanced aggregation created {len(event_agg)} events with {len(event_agg.columns)} features")
    return event_agg

In [19]:
# ==============================================================================
# IMPROVED: 5. Domain-Specific Physics & Risk Features
# ==============================================================================
def add_physics_features_advanced(df: pd.DataFrame) -> pd.DataFrame:
    """
    IMPROVED physics features with collision-specific indicators and uncertainty metrics.
    Includes Mahalanobis distance, risk trajectory, orbital decay rate, etc.
    """
    df = df.copy()
    
    # ----- Collision Geometry Features -----
    # Speed-to-distance ratio (higher = more risky approach)
    df['speed_to_distance_mean'] = (
        df['relative_speed_mean'] / (df['miss_distance_mean'] + 1e-9)
    )
    df['speed_to_distance_max'] = (
        df['relative_speed_max'] / (df['miss_distance_min'] + 1e-9)
    )
    
    # Position spread relative to miss distance
    total_position_uncertainty = (
        df['t_sigma_r_mean'] + df['c_sigma_r_mean'] +
        df['t_sigma_t_mean'] + df['c_sigma_t_mean'] +
        df['t_sigma_n_mean'] + df['c_sigma_n_mean']
    )
    df['uncertainty_to_distance'] = (
        total_position_uncertainty / (df['miss_distance_mean'] + 1e-9)
    )
    
    # ----- Covariance-Based Risk Indicators -----
    # Combined covariance determinant (larger = less certain)
    df['combined_cov_det_mean'] = (
        df['t_position_covariance_det_mean'] * df['c_position_covariance_det_mean']
    ) ** 0.5
    df['combined_cov_det_max'] = (
        df['t_position_covariance_det_max'] * df['c_position_covariance_det_max']
    ) ** 0.5
    
    # Covariance growth rate (uncertainty increasing?)
    df['cov_growth_rate'] = (
        (df['t_position_covariance_det_max'] - df['t_position_covariance_det_min']) /
        (df['t_position_covariance_det_mean'] + 1e-9)
    )
    
    # ----- Orbital Parameter Risk -----
    delta_sma = np.abs(df['t_j2k_sma_mean'] - df['c_j2k_sma_mean'])
    delta_ecc = np.abs(df['t_j2k_ecc_mean'] - df['c_j2k_ecc_mean'])
    delta_inc = np.abs(df['t_j2k_inc_mean'] - df['c_j2k_inc_mean'])
    
    df['orbital_distance'] = (delta_sma / (1e3 + 1e-9) +  # SMA difference in km
                              delta_ecc + delta_inc)
    
    # ----- Relative Velocity Profile -----
    # Relative speed components combine
    radial_speed = df['relative_velocity_r_mean'] ** 2
    tangential_speed = df['relative_velocity_t_mean'] ** 2
    normal_speed = df['relative_velocity_n_mean'] ** 2
    
    df['velocity_component_spread'] = (
        (radial_speed + tangential_speed + normal_speed) / (df['relative_speed_mean'] ** 2 + 1e-9)
    )
    
    # ----- Risk Trajectory Slope -----
    # If aggregation includes 'trend', use it; else estimate from std
    if 'risk_trend' in df.columns:
        df['risk_trajectory'] = df['risk_trend']
    else:
        # Estimate: high std relative to mean = high variability in risk
        df['risk_trajectory'] = 0  # Placeholder if trend not available
    
    # ----- Environmental & Space Weather -----
    df['solar_activity_mean'] = (df['F10_mean'] + df['F3M_mean']) / 2
    df['geomagnetic_activity'] = df['AP_mean']
    
    # Interaction terms
    df['activity_to_uncertainty'] = (
        df['solar_activity_mean'] * df['uncertainty_to_distance']
    )
    
    # ----- Renormalize for numerical stability -----
    large_value_cols = ['combined_cov_det_mean', 'combined_cov_det_max', 'cov_growth_rate']
    for col in large_value_cols:
        if col in df.columns:
            df[col] = np.log10(df[col] + 1)  # Log-compress extreme values
    
    logger.info(f"Advanced physics features added. Total features now: {df.shape[1]}")
    return df

In [20]:
# ----------------------------------------------------------------------
# 6. Categorical encoding (one‑hot for XGBoost, native for LightGBM) --
# ----------------------------------------------------------------------
def encode_categoricals(df: pd.DataFrame, cat_cols: List[str]) -> pd.DataFrame:
    """
    One‑hot encode the supplied categorical columns.
    The function returns a new DataFrame with the original categorical
    columns removed and the new dummy columns appended.
    """
    encoder = OneHotEncoder(sparse=False, drop="first", dtype=int)
    cat_matrix = encoder.fit_transform(df[cat_cols])

    # Build column names like c_object_type_1, mission_id_5 …
    cat_feature_names = encoder.get_feature_names_out(cat_cols)

    cat_df = pd.DataFrame(cat_matrix, columns=cat_feature_names, index=df.index)

    # Drop original categorical columns and concat the dummies
    df = df.drop(columns=cat_cols).join(cat_df)
    return df


In [21]:

# ----------------------------------------------------------------------
# 7. Log‑transform skewed numeric features -------------------------------
# ----------------------------------------------------------------------
def log_transform(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """
    Apply np.log1p to each column in `cols`.  The target column `risk` is
    **not** transformed because it is already in log10 scale.
    """
    for c in cols:
        df[c] = np.log1p(df[c])
    return df


In [22]:
# ==============================================================================
# IMPROVED: 7. Main Preprocessing Pipeline with Advanced Aggregation
# ==============================================================================
def get_preprocessed_event_table_advanced():
    """
    IMPROVED preprocessing: Uses advanced aggregation and physics features.
    Steps:
    1. Load raw data from Hugging Face
    2. Temporal filter (2-day cutoff before TCA)
    3. Advanced event aggregation (mean/std/min/max + percentiles/trends)
    4. Advanced physics feature extraction
    5. Categorical encoding
    6. Log transformation of skewed features
    7. Missing value handling
    8. Returns final feature table
    """
    logger.info("=" * 80)
    logger.info("STARTING IMPROVED PREPROCESSING PIPELINE")
    logger.info("=" * 80)
    
    # Step 1: Load
    logger.info("Step 1: Loading raw data...")
    df = load_raw()
    initial_rows = len(df)
    logger.info(f"Loaded {initial_rows} observations")
    
    # Step 2: Temporal filter
    logger.info("Step 2: Applying temporal filter...")
    df = filter_by_time(df)
    after_filter = len(df)
    logger.info(f"After 2-day cutoff: {after_filter} observations ({after_filter/initial_rows*100:.1f}%)")
    
    # Step 3: Advanced aggregation
    logger.info("Step 3: Advanced event aggregation...")
    event_df = aggregate_event_advanced(df)
    logger.info(f"Aggregated to {len(event_df)} events with {len(event_df.columns)} features")
    
    # Step 4: Advanced physics features
    logger.info("Step 4: Adding advanced physics features...")
    event_df = add_physics_features_advanced(event_df)
    logger.info(f"After physics features: {event_df.shape[1]} total features")
    
    # Step 5: Log transform skewed features
    logger.info("Step 5: Applying log transformations...")
    event_df = log_transform(event_df)
    
    # Step 6: Handle missing values
    logger.info("Step 6: Handling missing values...")
    event_df = event_df.fillna(event_df.mean(numeric_only=True))
    event_df = event_df.replace([np.inf, -np.inf], np.nan)
    event_df = event_df.fillna(event_df.mean(numeric_only=True))
    logger.info(f"Missing values after fillna: {event_df.isnull().sum().sum()}")
    
    # Step 7: Feature validation
    logger.info("Step 7: Validating features...")
    has_nan = event_df.isnull().sum().sum()
    has_inf = np.isinf(event_df.select_dtypes(include=[np.number])).sum().sum()
    
    if has_nan > 0 or has_inf > 0:
        logger.warning(f"NaN values: {has_nan}, Inf values: {has_inf} - attempting recovery...")
        event_df = event_df.fillna(0)
        event_df = event_df.replace([np.inf, -np.inf], 0)
    
    logger.info(f"FINAL: {len(event_df)} events × {event_df.shape[1]} features")
    
    # Save preprocessed data
    output_dir = Path("outputs")
    output_dir.mkdir(exist_ok=True)
    event_df.to_parquet(output_dir / "preprocessed_data_advanced.parquet", index=False)
    logger.info(f"Saved to outputs/preprocessed_data_advanced.parquet")
    
    return event_df

In [ ]:


# ----------------------------------------------------------------------
# 9. Example usage -------------------------------------------------------
# ----------------------------------------------------------------------
if __name__ == "__main__":
    # Run the pipeline and store the result
    preprocessed = get_preprocessed_event_table()
    print("Pre‑processed shape :", preprocessed.shape)
    print("First few rows:")
    print(preprocessed.head())

    # If you want to persist the table for later modelling:
    # Create the 'data' directory if it doesn't exist
    output_dir = "data"
    os.makedirs(output_dir, exist_ok=True)
    preprocessed.to_parquet(os.path.join(output_dir, "event_level_preprocessed.parquet"), index=False)

Pre‑processed shape : (11942, 112)
First few rows:
   event_id  relative_position_r_mean  relative_position_r_std  \
0         1                 10.325000                89.664314   
1         2               -767.450000               166.266347   
2         3                 29.386667                30.199997   
3         4                174.900000                 9.959418   
4         5                -10.242857                13.694508   

   relative_position_r_min  relative_position_r_max  relative_position_t_mean  \
0                    -82.0                     99.0              -6978.200000   
1                  -1161.1                   -686.8              -5358.150000   
2                      9.8                    136.7              11155.546667   
3                    163.6                    182.4             -19276.500000   
4                    -32.0                     12.3               -374.414286   

   relative_position_t_std  relative_position_t_min  relative_pos

/tmp/ipython-input-3904939813.py:123: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [10]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [23]:
# ==============================================================================
# IMPROVED: 8. Intelligent Feature Scaling
# ==============================================================================
class IntelligentScaler:
    """
    Applies different scalers to different feature groups:
    - StandardScaler: Normal features
    - RobustScaler: Features with outliers (covariance, large values)
    - QuantileTransformer: Highly skewed features
    - MinMaxScaler: Bounded features
    """
    def __init__(self):
        self.scalers = {}
        self.feature_groups = {}
    
    def fit(self, X, y=None):
        """Fit scalers to different feature groups based on statistical properties."""
        feature_stats = {}
        
        for col in X.columns:
            if col == 'event_id' or col == 'risk':
                continue
            
            series = X[col].dropna()
            if len(series) == 0:
                continue
            
            # Calculate statistics
            q1, q3 = series.quantile([0.25, 0.75])
            iqr = q3 - q1
            outliers = ((series < q1 - 1.5*iqr) | (series > q3 + 1.5*iqr)).sum()
            outlier_ratio = outliers / len(series) if len(series) > 0 else 0
            skewness = series.skew()
            
            feature_stats[col] = {
                'outlier_ratio': outlier_ratio,
                'skewness': np.abs(skewness),
                'min': series.min(),
                'max': series.max(),
                'std': series.std()
            }
        
        # Assign features to scaler groups
        for col, stats in feature_stats.items():
            if stats['outlier_ratio'] > 0.05:  # >5% outliers
                group = 'robust'
            elif stats['skewness'] > 1.5:  # Highly skewed
                group = 'quantile'
            elif stats['min'] >= 0 and stats['max'] > 0 and stats['std'] < stats['max']:  # Bounded
                group = 'minmax'
            else:
                group = 'standard'
            
            self.feature_groups[col] = group
        
        # Create scalers for each group
        self.scalers['standard'] = StandardScaler()
        self.scalers['robust'] = RobustScaler(quantile_range=(10, 90))
        self.scalers['quantile'] = QuantileTransformer(n_quantiles=1000, output_distribution='uniform', 
                                                       random_state=42)
        self.scalers['minmax'] = MinMaxScaler(feature_range=(0, 1))
        
        # Fit each scaler with its respective features
        for group_name in ['standard', 'robust', 'quantile', 'minmax']:
            group_cols = [col for col, grp in self.feature_groups.items() if grp == group_name]
            if group_cols:
                self.scalers[group_name].fit(X[group_cols])
        
        logger.info(f"IntelligentScaler fitted: {len(self.feature_groups)} features")
        logger.info(f"  Standard: {len([g for g in self.feature_groups.values() if g == 'standard'])}")
        logger.info(f"  Robust: {len([g for g in self.feature_groups.values() if g == 'robust'])}")
        logger.info(f"  Quantile: {len([g for g in self.feature_groups.values() if g == 'quantile'])}")
        logger.info(f"  MinMax: {len([g for g in self.feature_groups.values() if g == 'minmax'])}")
        
        return self
    
    def transform(self, X):
        """Apply appropriate scaler to each feature."""
        X_scaled = X.copy()
        
        for group_name in ['standard', 'robust', 'quantile', 'minmax']:
            group_cols = [col for col, grp in self.feature_groups.items() if grp == group_name]
            if group_cols and group_name in self.scalers:
                X_scaled[group_cols] = self.scalers[group_name].transform(X[group_cols])
        
        return X_scaled
    
    def fit_transform(self, X, y=None):
        """Fit and transform in one step."""
        return self.fit(X, y).transform(X)


def prepare_train_test_data_advanced(event_df):
    """
    IMPROVED: Prepares train/test split with intelligent scaling.
    Uses stratified split by risk level and applies IntelligentScaler.
    """
    logger.info("=" * 80)
    logger.info("PREPARING TRAIN/TEST DATA WITH INTELLIGENT SCALING")
    logger.info("=" * 80)
    
    # Define high-risk threshold
    risk_threshold = -6  # log10 scale
    y_binary = (event_df['risk'] >= risk_threshold).astype(int)
    
    # Select features (drop risk and event_id)
    feature_cols = [col for col in event_df.columns if col not in ['risk', 'event_id']]
    X = event_df[feature_cols].copy()
    y = event_df['risk'].copy()
    
    logger.info(f"Input shape: X={X.shape}, y={y.shape}")
    logger.info(f"Risk distribution:")
    logger.info(f"  Low-risk (< {risk_threshold}): {(y_binary == 0).sum()} ({(y_binary == 0).sum()/len(y)*100:.1f}%)")
    logger.info(f"  High-risk (>= {risk_threshold}): {(y_binary == 1).sum()} ({(y_binary == 1).sum()/len(y)*100:.1f}%)")
    
    # Stratified train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y_binary
    )
    
    logger.info(f"\nTrain set: {X_train.shape[0]} samples")
    logger.info(f"Test set: {X_test.shape[0]} samples")
    
    # Apply intelligent scaling
    scaler = IntelligentScaler()
    scaler.fit(X_train)
    
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    logger.info(f"\nScaling applied successfully")
    logger.info(f"Train: min={X_train_scaled.min().min():.4f}, max={X_train_scaled.max().max():.4f}")
    logger.info(f"Test: min={X_test_scaled.min().min():.4f}, max={X_test_scaled.max().max():.4f}")
    
    return {
        'X_train': X_train_scaled,
        'X_test': X_test_scaled,
        'y_train': y_train.values,
        'y_test': y_test.values,
        'scaler': scaler,
        'feature_names': feature_cols,
        'y_binary_train': y_binary[X_train.index].values,
        'y_binary_test': y_binary[X_test.index].values
    }

In [24]:
# ==============================================================================
# 11. Model Training Pipeline
# ==============================================================================
def prepare_train_test_data(df: pd.DataFrame, test_size=0.2, random_state=42):
    """
    Prepare training and testing datasets.
    - Separate target (risk) from features
    - Handle missing values and scale features
    - Return (X_train, X_test, y_train, y_test)
    """
    # Extract target and features
    if "risk" not in df.columns:
        raise ValueError("Target column 'risk' not found in DataFrame")
    
    y = df["risk"].copy()
    X = df.drop(columns=["risk", "event_id"] if "event_id" in df.columns else ["risk"])
    
    logger.info(f"Dataset shape: {X.shape}")
    logger.info(f"Target values range: [{y.min():.2e}, {y.max():.2e}]")
    logger.info(f"Missing values: {X.isnull().sum().sum()}")
    
    # Handle remaining NaNs
    X = X.fillna(0)
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)
    
    logger.info(f"Training set: {X_train_scaled.shape}")
    logger.info(f"Test set: {X_test_scaled.shape}")
    
    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

In [25]:
# ==============================================================================
# 12. Train Models: XGBoost and LightGBM
# ==============================================================================
def train_models(X_train, X_test, y_train, y_test):
    """
    Train XGBoost and LightGBM models with optimized params for this task.
    Returns both trained models and evaluation results.
    """
    try:
        import xgboost as xgb
        import lightgbm as lgb
    except ImportError as e:
        logger.error(f"Missing ML library: {e}")
        logger.info("Install with: pip install xgboost lightgbm")
        raise
    
    results_df = []
    models = {}
    
    # ========================================================================
    # XGBoost
    # ========================================================================
    logger.info("Training XGBoost model...")
    xgb_model = xgb.XGBRegressor(
        objective="reg:squarederror",
        max_depth=7,
        learning_rate=0.05,
        n_estimators=500,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=0
    )
    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    y_pred_xgb = xgb_model.predict(X_test)
    results_xgb = evaluate_model(y_test.values, y_pred_xgb, "XGBoost")
    results_df.append(results_xgb)
    models["XGBoost"] = xgb_model
    
    # ========================================================================
    # LightGBM
    # ========================================================================
    logger.info("Training LightGBM model...")
    lgb_model = lgb.LGBMRegressor(
        objective="regression",
        max_depth=7,
        learning_rate=0.05,
        n_estimators=500,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )
    lgb_model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[lgb.log_evaluation(period=0)]  # Suppress verbose output
    )
    
    y_pred_lgb = lgb_model.predict(X_test)
    results_lgb = evaluate_model(y_test.values, y_pred_lgb, "LightGBM")
    results_df.append(results_lgb)
    models["LightGBM"] = lgb_model
    
    # Create results summary
    results_summary = pd.DataFrame(results_df)
    
    logger.info("\n" + "="*60)
    logger.info("MODEL COMPARISON SUMMARY")
    logger.info("="*60)
    logger.info(results_summary.to_string(index=False))
    logger.info("="*60 + "\n")
    
    return models, results_summary, {
        "y_pred_xgb": y_pred_xgb,
        "y_pred_lgb": y_pred_lgb,
        "y_test": y_test.values
    }

In [26]:
# ==============================================================================
# 13. Feature Importance Analysis
# ==============================================================================
def plot_feature_importance(models, X_train, top_n=20):
    """
    Plot feature importance from both XGBoost and LightGBM models.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        logger.warning("matplotlib not available - skipping plots")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for idx, (model_name, model) in enumerate(models.items()):
        importances = model.feature_importances_
        feature_names = X_train.columns
        
        # Sort and get top features
        indices = np.argsort(importances)[-top_n:]
        top_importances = importances[indices]
        top_features = feature_names[indices]
        
        # Plot
        axes[idx].barh(range(len(top_features)), top_importances, color='steelblue')
        axes[idx].set_yticks(range(len(top_features)))
        axes[idx].set_yticklabels(top_features, fontsize=9)
        axes[idx].set_xlabel("Importance Score", fontsize=11)
        axes[idx].set_title(f"{model_name} - Top {top_n} Features", fontsize=12, fontweight='bold')
        axes[idx].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("feature_importance.png", dpi=150, bbox_inches='tight')
    logger.info("Feature importance plot saved: feature_importance.png")
    plt.show()

In [27]:
# ==============================================================================
# 14. Prediction Analysis and Visualization
# ==============================================================================
def visualize_predictions(predictions_dict):
    """
    Visualize predicted vs actual risk values and error distributions.
    """
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        logger.warning("matplotlib not available - skipping plots")
        return
    
    y_test = predictions_dict["y_test"]
    y_pred_xgb = predictions_dict["y_pred_xgb"]
    y_pred_lgb = predictions_dict["y_pred_lgb"]
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # ---- XGBoost: Predicted vs Actual ----
    ax = axes[0, 0]
    ax.scatter(y_test, y_pred_xgb, alpha=0.5, s=20)
    min_val = min(y_test.min(), y_pred_xgb.min())
    max_val = max(y_test.max(), y_pred_xgb.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel("Actual Risk", fontsize=11)
    ax.set_ylabel("Predicted Risk (XGBoost)", fontsize=11)
    ax.set_title("XGBoost: Predicted vs Actual", fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # ---- LightGBM: Predicted vs Actual ----
    ax = axes[0, 1]
    ax.scatter(y_test, y_pred_lgb, alpha=0.5, s=20, color='orange')
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel("Actual Risk", fontsize=11)
    ax.set_ylabel("Predicted Risk (LightGBM)", fontsize=11)
    ax.set_title("LightGBM: Predicted vs Actual", fontsize=12, fontweight='bold')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.legend()
    ax.grid(alpha=0.3)
    
    # ---- XGBoost: Error Distribution (Log Scale) ----
    ax = axes[1, 0]
    error_xgb = np.abs(y_test - y_pred_xgb)
    ax.hist(np.log10(error_xgb + 1e-9), bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_xlabel("log₁₀(Absolute Error)", fontsize=11)
    ax.set_ylabel("Frequency", fontsize=11)
    ax.set_title("XGBoost: Log Error Distribution", fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    
    # ---- LightGBM: Error Distribution (Log Scale) ----
    ax = axes[1, 1]
    error_lgb = np.abs(y_test - y_pred_lgb)
    ax.hist(np.log10(error_lgb + 1e-9), bins=50, color='orange', alpha=0.7, edgecolor='black')
    ax.set_xlabel("log₁₀(Absolute Error)", fontsize=11)
    ax.set_ylabel("Frequency", fontsize=11)
    ax.set_title("LightGBM: Log Error Distribution", fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig("prediction_analysis.png", dpi=150, bbox_inches='tight')
    logger.info("Prediction analysis plot saved: prediction_analysis.png")
    plt.show()
    
    # Print high-risk events analysis
    threshold = 1e-6
    high_risk_actual = y_test >= threshold
    
    logger.info(f"\n{'='*60}")
    logger.info("HIGH-RISK EVENTS ANALYSIS (Risk >= 1e-6)")
    logger.info(f"{'='*60}")
    logger.info(f"Actual high-risk events: {high_risk_actual.sum()} / {len(y_test)}")
    logger.info(f"High-risk percentage: {100*high_risk_actual.sum()/len(y_test):.3f}%")
    logger.info(f"{'='*60}\n")

In [29]:
# ==============================================================================
# 15. MAIN EXECUTION: Complete Pipeline
# ==============================================================================
if __name__ == "__main__":
    logger.info("="*80)
    logger.info("SCRAP: SATELLITE COLLISION RISK ASSESSMENT AND PREDICTION")
    logger.info("="*80 + "\n")
    
    # ---- Step 1: Preprocess Data ----
    logger.info("STEP 1: Preprocessing and Feature Engineering")
    logger.info("-"*80)
    preprocessed_df = get_preprocessed_event_table()
    logger.info(f"Preprocessed data shape: {preprocessed_df.shape}\n")
    
    # ---- Step 2: Prepare Train/Test Split ----
    logger.info("STEP 2: Train-Test Split and Scaling")
    logger.info("-"*80)
    X_train, X_test, y_train, y_test, scaler = prepare_train_test_data(preprocessed_df)
    logger.info(f"X_train shape: {X_train.shape}")
    logger.info(f"X_test shape: {X_test.shape}\n")
    
    # ---- Step 3: Train Models ----
    logger.info("STEP 3: Model Training")
    logger.info("-"*80)
    models, results_summary, predictions_dict = train_models(X_train, X_test, y_train, y_test)
    
    # ---- Step 4: Feature Importance ----
    logger.info("STEP 4: Feature Importance Analysis")
    logger.info("-"*80)
    plot_feature_importance(models, X_train, top_n=20)
    
    # ---- Step 5: Prediction Visualization ----
    logger.info("STEP 5: Prediction Analysis and Visualization")
    logger.info("-"*80)
    visualize_predictions(predictions_dict)
    
    # ---- Step 6: Save Models and Results ----
    logger.info("STEP 6: Saving Models and Results")
    logger.info("-"*80)
    output_dir = "outputs"
    os.makedirs(output_dir, exist_ok=True)
    
    import pickle
    with open(os.path.join(output_dir, "models.pkl"), "wb") as f:
        pickle.dump(models, f)
    
    results_summary.to_csv(os.path.join(output_dir, "model_results.csv"), index=False)
    preprocessed_df.to_parquet(os.path.join(output_dir, "preprocessed_data.parquet"), index=False)
    
    logger.info(f"Models saved to: {output_dir}/models.pkl")
    logger.info(f"Results saved to: {output_dir}/model_results.csv")
    logger.info(f"Preprocessed data saved to: {output_dir}/preprocessed_data.parquet")
    
    logger.info("\n" + "="*80)
    logger.info("PIPELINE COMPLETE!")
    logger.info("="*80)

2026-03-01 22:11:36,165 - INFO - ================================================================================
2026-03-01 22:11:36,166 - INFO - SCRAP: SATELLITE COLLISION RISK ASSESSMENT AND PREDICTION
2026-03-01 22:11:36,167 - INFO - ================================================================================

2026-03-01 22:11:36,168 - INFO - STEP 1: Preprocessing and Feature Engineering
2026-03-01 22:11:36,169 - INFO - --------------------------------------------------------------------------------


NameError: name 'get_preprocessed_event_table' is not defined

In [ ]:
# Optional dependencies for this section:
# !pip install -U pulp plotly

import math
import random
import matplotlib.pyplot as plt

try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except Exception:
    HAS_PLOTLY = False

try:
    from pulp import (
        LpProblem,
        LpMaximize,
        LpVariable,
        LpBinary,
        lpSum,
        LpStatus,
        PULP_CBC_CMD,
        value,
    )
    HAS_PULP = True
except Exception:
    HAS_PULP = False

random.seed(42)
np.random.seed(42)


In [6]:
def angular_sep(lat1, lon1, lat2, lon2):
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dlambda = math.radians(lon2 - lon1)
    a = (
        math.sin((phi2 - phi1) / 2) ** 2
        + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    )
    return 2 * math.degrees(math.asin(min(1.0, math.sqrt(a))))

def wrap_longitude(series: pd.Series) -> pd.Series:
    lon = pd.to_numeric(series, errors="coerce")
    return ((lon + 180.0) % 360.0) - 180.0

def build_satellite_positions_from_scrap(
    scrap_df: pd.DataFrame,
    num_sats: int = 24,
    time_slots: int = 6,
    random_state: int = 42,
):
    rng = np.random.default_rng(random_state)
    sat_positions = {}
    sat_ids = list(range(num_sats))

    inc_pool = pd.to_numeric(
        scrap_df.get("t_j2k_inc", pd.Series(dtype=float)), errors="coerce"
    ).dropna().to_numpy()
    if inc_pool.size == 0:
        inc_pool = rng.uniform(30, 98, size=max(num_sats, 128))

    phase_offsets = rng.uniform(0, 2 * np.pi, size=num_sats)
    for sat_idx, sat_id in enumerate(sat_ids):
        inc_rad = np.radians(float(inc_pool[sat_idx % len(inc_pool)]))
        phase = phase_offsets[sat_idx]
        for slot in range(time_slots):
            theta = (2 * np.pi * slot / time_slots) + phase
            lat = np.degrees(np.arcsin(np.sin(inc_rad) * np.sin(theta)))
            lon = ((np.degrees(theta * 1.8) + sat_idx * 15 + 180) % 360) - 180
            sat_positions[(sat_id, slot)] = {"lat": float(lat), "lon": float(lon)}

    return sat_positions, sat_ids

def build_tasks_from_scrap(
    scrap_df: pd.DataFrame,
    num_tasks: int = 40,
    random_state: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)

    if "geocentric_latitude" in scrap_df.columns:
        lat = pd.to_numeric(scrap_df["geocentric_latitude"], errors="coerce")
    elif "t_j2k_inc" in scrap_df.columns:
        lat = pd.to_numeric(scrap_df["t_j2k_inc"], errors="coerce").clip(-90, 90)
    else:
        lat = pd.Series(rng.uniform(-60, 60, size=len(scrap_df)))

    if "geocentric_longitude" in scrap_df.columns:
        lon = wrap_longitude(scrap_df["geocentric_longitude"])
    elif "relative_position_t" in scrap_df.columns:
        lon = wrap_longitude(scrap_df["relative_position_t"])
    else:
        lon = pd.Series(rng.uniform(-180, 180, size=len(scrap_df)))

    if "risk" in scrap_df.columns:
        priority_raw = pd.to_numeric(scrap_df["risk"], errors="coerce")
    else:
        priority_raw = pd.Series(rng.uniform(0.2, 1.0, size=len(scrap_df)))

    tasks = pd.DataFrame({
        "lat": lat,
        "lon": lon,
        "priority_raw": priority_raw,
    }).dropna()

    if tasks.empty:
        tasks = pd.DataFrame({
            "lat": rng.uniform(-60, 60, size=num_tasks),
            "lon": rng.uniform(-180, 180, size=num_tasks),
            "priority_raw": rng.uniform(0.2, 1.0, size=num_tasks),
        })

    n = min(num_tasks, len(tasks))
    tasks = tasks.sample(n=n, random_state=random_state).reset_index(drop=True)

    p = tasks["priority_raw"]
    span = (p.max() - p.min()) if p.max() > p.min() else 1.0
    tasks["priority"] = 1.0 + 9.0 * ((p - p.min()) / span)
    tasks["task_id"] = tasks.index.astype(int)
    return tasks[["task_id", "lat", "lon", "priority"]]

def build_eligibility(
    sat_positions: dict,
    tasks_df: pd.DataFrame,
    sat_ids: list,
    time_slots: int,
    footprint_ang_deg: float = 20.0,
) -> dict:
    eligibility = {}
    for sat in sat_ids:
        for slot in range(time_slots):
            pos = sat_positions.get((sat, slot))
            if pos is None:
                continue
            for row in tasks_df.itertuples(index=False):
                eligibility[(sat, slot, int(row.task_id))] = (
                    angular_sep(pos["lat"], pos["lon"], row.lat, row.lon) <= footprint_ang_deg
                )
    return eligibility

def simulate_unavailable_sat_slots(
    sat_ids: list,
    time_slots: int,
    anomaly_rate: float = 0.05,
    random_state: int = 42,
) -> set:
    rng = random.Random(random_state)
    unavailable = set()
    for sat in sat_ids:
        for slot in range(time_slots):
            if rng.random() < anomaly_rate:
                unavailable.add((sat, slot))
    return unavailable

def solve_schedule(
    tasks_df: pd.DataFrame,
    sat_ids: list,
    time_slots: int,
    eligibility: dict,
    unavailable: set,
) -> tuple[pd.DataFrame, object]:
    if not HAS_PULP:
        raise ImportError("PuLP is not installed. Run: pip install pulp")

    priorities = tasks_df.set_index("task_id")["priority"].to_dict()
    task_lookup = tasks_df.set_index("task_id")

    prob = LpProblem("satellite_scheduler", LpMaximize)
    x = {}
    for sat in sat_ids:
        for slot in range(time_slots):
            if (sat, slot) in unavailable:
                continue
            for tid in tasks_df["task_id"]:
                if eligibility.get((sat, slot, int(tid)), False):
                    x[(sat, slot, int(tid))] = LpVariable(
                        f"x_{sat}_{slot}_{int(tid)}",
                        lowBound=0,
                        upBound=1,
                        cat=LpBinary,
                    )

    if not x:
        raise ValueError("No feasible assignments were generated.")

    prob += lpSum(priorities[tid] * var for (_, _, tid), var in x.items())

    for tid in tasks_df["task_id"]:
        prob += lpSum(var for (s, t, k), var in x.items() if k == int(tid)) <= 1

    for sat in sat_ids:
        for slot in range(time_slots):
            prob += lpSum(var for (s, t, k), var in x.items() if s == sat and t == slot) <= 1

    prob.solve(PULP_CBC_CMD(msg=False))

    assignments = []
    for (sat, slot, tid), var in x.items():
        v = value(var)
        if v is not None and v > 0.5:
            task = task_lookup.loc[tid]
            assignments.append({
                "sat_id": sat,
                "slot": slot,
                "task_id": int(tid),
                "task_lat": float(task["lat"]),
                "task_lon": float(task["lon"]),
                "priority": float(task["priority"]),
            })

    schedule_df = pd.DataFrame(assignments)
    if not schedule_df.empty:
        schedule_df = schedule_df.sort_values(["slot", "sat_id"]).reset_index(drop=True)

    return schedule_df, prob

def plot_schedule_2d(
    tasks_df: pd.DataFrame,
    sat_positions: dict,
    sat_ids: list,
    slot_to_plot: int = 0,
    schedule_df: pd.DataFrame | None = None,
):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.scatter(tasks_df["lon"], tasks_df["lat"], marker="o", alpha=0.8, label="Tasks")

    sat_lons = [sat_positions[(s, slot_to_plot)]["lon"] for s in sat_ids if (s, slot_to_plot) in sat_positions]
    sat_lats = [sat_positions[(s, slot_to_plot)]["lat"] for s in sat_ids if (s, slot_to_plot) in sat_positions]
    ax.scatter(sat_lons, sat_lats, marker="^", s=80, color="tab:red", label=f"Satellites (slot {slot_to_plot})")

    if schedule_df is not None and not schedule_df.empty:
        task_lookup = tasks_df.set_index("task_id")
        chosen = schedule_df[schedule_df["slot"] == slot_to_plot]
        for row in chosen.itertuples(index=False):
            sp = sat_positions.get((int(row.sat_id), slot_to_plot))
            if sp is None or int(row.task_id) not in task_lookup.index:
                continue
            tp = task_lookup.loc[int(row.task_id)]
            ax.plot([sp["lon"], tp["lon"]], [sp["lat"], tp["lat"]], color="gray", alpha=0.5)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("Satellite Tasking (2D Map)")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    return fig

def _latlon_to_xyz(lat_deg: float, lon_deg: float, radius_km: float):
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    x = radius_km * np.cos(lat) * np.cos(lon)
    y = radius_km * np.cos(lat) * np.sin(lon)
    z = radius_km * np.sin(lat)
    return x, y, z

def plot_orbits_3d(sat_positions: dict, sat_ids: list, time_slots: int, orbit_alt_km: float = 550.0):
    if not HAS_PLOTLY:
        raise ImportError("Plotly is not installed. Run: pip install plotly")

    earth_r = 6371.0
    lats = np.linspace(-np.pi / 2, np.pi / 2, 60)
    lons = np.linspace(-np.pi, np.pi, 120)
    xs = earth_r * np.outer(np.cos(lats), np.cos(lons))
    ys = earth_r * np.outer(np.cos(lats), np.sin(lons))
    zs = earth_r * np.outer(np.sin(lats), np.ones_like(lons))

    traces = [
        go.Surface(
            x=xs,
            y=ys,
            z=zs,
            showscale=False,
            colorscale="Blues",
            opacity=0.85,
            name="Earth",
            hoverinfo="skip",
        )
    ]

    for sat in sat_ids:
        x_orbit, y_orbit, z_orbit = [], [], []
        for slot in range(time_slots):
            p = sat_positions.get((sat, slot))
            if p is None:
                continue
            x, y, z = _latlon_to_xyz(p["lat"], p["lon"], earth_r + orbit_alt_km)
            x_orbit.append(x)
            y_orbit.append(y)
            z_orbit.append(z)

        if len(x_orbit) >= 2:
            traces.append(
                go.Scatter3d(
                    x=x_orbit,
                    y=y_orbit,
                    z=z_orbit,
                    mode="lines+markers",
                    marker=dict(size=2),
                    line=dict(width=2),
                    name=f"SAT-{sat}",
                )
            )

    fig = go.Figure(data=traces)
    fig.update_layout(
        title="3D Earth: Satellite Orbit Tracks",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode="data",
        ),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


In [7]:
TIME_SLOTS_SCHED = 6
FOOTPRINT_ANG_DEG = 20.0

scrap_df_for_sched = data.copy() if "data" in globals() else load_raw()

sat_positions, sat_ids = build_satellite_positions_from_scrap(
    scrap_df_for_sched,
    num_sats=24,
    time_slots=TIME_SLOTS_SCHED,
    random_state=42,
)
tasks_df = build_tasks_from_scrap(scrap_df_for_sched, num_tasks=40, random_state=42)
eligibility = build_eligibility(
    sat_positions,
    tasks_df,
    sat_ids,
    time_slots=TIME_SLOTS_SCHED,
    footprint_ang_deg=FOOTPRINT_ANG_DEG,
)
unavailable = simulate_unavailable_sat_slots(
    sat_ids,
    time_slots=TIME_SLOTS_SCHED,
    anomaly_rate=0.05,
    random_state=42,
)

schedule_df, sched_prob = solve_schedule(
    tasks_df,
    sat_ids,
    time_slots=TIME_SLOTS_SCHED,
    eligibility=eligibility,
    unavailable=unavailable,
)

print(f"Solver status: {LpStatus[sched_prob.status]}")
print(f"Total tasks: {len(tasks_df)}")
print(f"Scheduled tasks: {len(schedule_df)}")
schedule_df.head(10)


NameError: name 'math' is not defined

In [ ]:
plot_schedule_2d(
    tasks_df=tasks_df,
    sat_positions=sat_positions,
    sat_ids=sat_ids,
    slot_to_plot=0,
    schedule_df=schedule_df,
)


In [11]:
# ==============================================================================
# REGENERATE ESA SCORES IF KERNEL WAS RESTARTED
# ==============================================================================
# Load saved outputs from the previous successful run
import pickle
import os

output_dir = "outputs"
models_path = os.path.join(output_dir, "models.pkl")
data_path = os.path.join(output_dir, "preprocessed_data.parquet")

if os.path.exists(models_path) and os.path.exists(data_path):
    logger.info("Loading previously trained models and data...")
    
    # Load models
    with open(models_path, "rb") as f:
        models = pickle.load(f)
        xgb_model = models["XGBoost"]
        lgb_model = models["LightGBM"]
    
    # Load preprocessed data
    preprocessed_df = pd.read_parquet(data_path)
    logger.info(f"Loaded preprocessed data: {preprocessed_df.shape}")
    
    # Regenerate train/test split (with same random state for reproducibility)
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    
    y = preprocessed_df["risk"].copy()
    X = preprocessed_df.drop(columns=["risk", "event_id"] if "event_id" in preprocessed_df.columns else ["risk"])
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    X_test = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)
    
    # Generate predictions
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_lgb = lgb_model.predict(X_test)
    
    predictions_dict = {
        "y_pred_xgb": y_pred_xgb,
        "y_pred_lgb": y_pred_lgb,
        "y_test": y_test.values
    }
    
    logger.info("✓ Models and data successfully loaded and predictions regenerated")
else:
    logger.warning(f"Could not find saved models at {models_path}")
    logger.warning("Please run the full training pipeline first")

2026-03-01 21:50:52,579 - INFO - Loading previously trained models and data...
2026-03-01 21:50:54,351 - INFO - Loaded preprocessed data: (11942, 112)
2026-03-01 21:50:54,468 - INFO - ✓ Models and data successfully loaded and predictions regenerated


In [ ]:
if HAS_PLOTLY:
    fig3d = plot_orbits_3d(
        sat_positions=sat_positions,
        sat_ids=sat_ids,
        time_slots=TIME_SLOTS_SCHED,
        orbit_alt_km=550.0,
    )
    fig3d.show()
else:
    print("Plotly not installed. Run: pip install plotly")


In [30]:
# ==============================================================================
# ESA OFFICIAL SCORING METRIC
# ==============================================================================
"""
ESA Collision Avoidance Challenge Scoring:

L(r, r^) = 1/F2 * MSE(r, r^)

Where:
- F2 is computed with β=2 over the whole dataset using two classes:
  * high-risk: r >= 10^-6 (which is -6 in log10 scale)
  * low-risk: r < 10^-6 (which is < -6 in log10 scale)
  
- MSE is ONLY computed for high-risk events (r >= 10^-6)

- F2 = (1 + β²) * (precision * recall) / (β² * precision + recall)
  with β = 2

The loss heavily penalizes FALSE NEGATIVES (missed high-risk collisions).
"""

def compute_esa_f2_score(y_true, y_pred, log_threshold=-6, beta=2):
    """
    Compute F-β score for the official challenge metric.
    
    Args:
        y_true: True risk values (in log10 scale)
        y_pred: Predicted risk values (in log10 scale)
        log_threshold: Risk threshold in log10 scale (default -6 = 1e-6)
        beta: Beta parameter for F-score (default 2, emphasizes recall)
    
    Returns:
        F2 score (higher is better)
    """
    # Convert to binary classification
    y_true_binary = (y_true >= log_threshold).astype(int)
    y_pred_binary = (y_pred >= log_threshold).astype(int)
    
    # Compute confusion matrix
    tp = np.sum((y_pred_binary == 1) & (y_true_binary == 1))  # True Positives
    fp = np.sum((y_pred_binary == 1) & (y_true_binary == 0))  # False Positives
    fn = np.sum((y_pred_binary == 0) & (y_true_binary == 1))  # False Negatives
    tn = np.sum((y_pred_binary == 0) & (y_true_binary == 0))  # True Negatives
    
    # Compute precision and recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # Compute F-β score
    beta_sq = beta ** 2
    if (beta_sq * precision + recall) == 0:
        return 0.0
    
    f_beta = (1 + beta_sq) * (precision * recall) / (beta_sq * precision + recall)
    
    return f_beta, precision, recall, tp, fp, fn, tn


def compute_esa_loss(y_true, y_pred, log_threshold=-6):
    """
    Compute official ESA challenge loss.
    
    L(r, r^) = 1/F2 * MSE(r, r^)
    
    Where MSE is only computed for high-risk events (r >= 1e-6).
    In log10 scale: r >= -6
    
    Args:
        y_true: True risk values (log10 scale)
        y_pred: Predicted risk values (log10 scale)
        log_threshold: Risk threshold in log10 scale (default -6)
    
    Returns:
        (ESA loss, F2 score, MSE for high-risk)
    """
    # Get F2 score and detailed metrics
    f2, precision, recall, tp, fp, fn, tn = compute_esa_f2_score(y_true, y_pred, 
                                                                   log_threshold=log_threshold, 
                                                                   beta=2)
    
    # Get high-risk mask
    high_risk_mask = y_true >= log_threshold
    
    # Compute MSE only for high-risk events
    if high_risk_mask.sum() > 0:
        mse_high_risk = np.mean((y_true[high_risk_mask] - y_pred[high_risk_mask]) ** 2)
    else:
        mse_high_risk = 0
    
    # Compute loss
    if f2 > 0:
        loss = 1.0 / f2 * mse_high_risk
    else:
        # If F2 is 0 (no true positives), loss is undefined; use large penalty
        loss = float('inf')
    
    return loss, f2, mse_high_risk, precision, recall, tp, fp, fn, tn


def evaluate_esa_challenge(y_true, y_pred, model_name="Model"):
    """
    Comprehensive evaluation using ESA challenge metrics.
    """
    logger.info(f"\n{'='*70}")
    logger.info(f"ESA COLLISION AVOIDANCE CHALLENGE EVALUATION: {model_name}")
    logger.info(f"{'='*70}")
    
    # Official ESA metric (threshold in log10 scale)
    log_threshold = -6  # log10(1e-6)
    loss, f2, mse_high, precision, recall, tp, fp, fn, tn = compute_esa_loss(y_true, y_pred, 
                                                                               log_threshold=log_threshold)
    
    # Standard metrics
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Class distribution info
    n_high_risk = (y_true >= log_threshold).sum()
    n_total = len(y_true)
    high_risk_pct = 100 * n_high_risk / n_total
    
    logger.info(f"")
    logger.info(f"OFFICIAL ESA SCORE:")
    logger.info(f"  Loss (L):           {loss:.6e}  (LOWER IS BETTER)")
    logger.info(f"")
    logger.info(f"COMPONENT METRICS:")
    logger.info(f"  F2-Score:           {f2:.6f}  (max=1.0, higher is better)")
    logger.info(f"  Precision:          {precision:.6f}  (TP / (TP + FP))")
    logger.info(f"  Recall:             {recall:.6f}  (TP / (TP + FN))")
    logger.info(f"  MSE (high-risk):    {mse_high:.6e}  (only for r >= 1e-6)")
    logger.info(f"")
    logger.info(f"CONFUSION MATRIX (threshold = 1e-6):")
    logger.info(f"  TP (correct high-risk):    {int(tp)}")
    logger.info(f"  FP (false alarms):         {int(fp)}")
    logger.info(f"  FN (missed collisions):    {int(fn)} ⚠️ CRITICAL - reduces F2")
    logger.info(f"  TN (correct low-risk):     {int(tn)}")
    logger.info(f"")
    logger.info(f"SUPPLEMENTARY METRICS:")
    logger.info(f"  MSE (all):          {mse:.6e}")
    logger.info(f"  MAE (all):          {mae:.6e}")
    logger.info(f"  R² Score:           {r2:.6f}")
    logger.info(f"")
    logger.info(f"CLASS DISTRIBUTION (threshold = 1e-6 = -6 in log10):")
    logger.info(f"  Data range:         [{y_true.min():.2f}, {y_true.max():.2f}]")
    logger.info(f"  High-risk events:   {n_high_risk} / {n_total}  ({high_risk_pct:.2f}%)")
    logger.info(f"  Low-risk events:    {n_total - n_high_risk} / {n_total}  ({100-high_risk_pct:.2f}%)")
    logger.info(f"{'='*70}\n")
    
    return {
        "model": model_name,
        "esa_loss": loss,
        "f2_score": f2,
        "precision": precision,
        "recall": recall,
        "mse_high_risk": mse_high,
        "mse_all": mse,
        "mae_all": mae,
        "r2_score": r2,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "n_high_risk": n_high_risk,
        "n_total": n_total,
        "high_risk_pct": high_risk_pct
    }


# ---- Compute ESA Scores for Both Models ----
logger.info("\n\n")
logger.info("╔" + "="*68 + "╗")
logger.info("║" + " "*15 + "ESA COLLISION CHALLENGE SCORING" + " "*23 + "║")
logger.info("╚" + "="*68 + "╝")

esa_results = []

# XGBoost
xgb_results = evaluate_esa_challenge(y_test.values, predictions_dict["y_pred_xgb"], "XGBoost")
esa_results.append(xgb_results)

# LightGBM
lgb_results = evaluate_esa_challenge(y_test.values, predictions_dict["y_pred_lgb"], "LightGBM")
esa_results.append(lgb_results)

# Comparison table
esa_df = pd.DataFrame(esa_results)
logger.info("\nMODEL COMPARISON (ESA CHALLENGE METRICS):")
logger.info("-" * 80)
comparison_cols = ["model", "esa_loss", "f2_score", "precision", "recall", "r2_score"]
for row in esa_df[comparison_cols].values:
    logger.info(f"  {row[0]:12} | Loss: {row[1]:12.6e} | F2: {row[2]:.6f} | "
                f"Prec: {row[3]:.4f} | Recall: {row[4]:.4f} | R²: {row[5]:.6f}")
logger.info("-" * 80)

# Identify best model based on loss
best_idx = esa_df["esa_loss"].idxmin()
best_model = esa_df.loc[best_idx, "model"]
best_loss = esa_df.loc[best_idx, "esa_loss"]
best_f2 = esa_df.loc[best_idx, "f2_score"]

logger.info(f"\n🏆 BEST MODEL: {best_model}")
logger.info(f"   Official ESA Loss:  {best_loss:.6e}")
logger.info(f"   F2-Score:           {best_f2:.6f}")
logger.info(f"   (Lower loss is better)\n")

# Save ESA results
esa_df.to_csv(os.path.join(output_dir, "esa_challenge_scores.csv"), index=False)
logger.info(f"✓ ESA scores saved to: {output_dir}/esa_challenge_scores.csv\n")

2026-03-01 22:11:52,602 - INFO - 


2026-03-01 22:11:52,604 - INFO - ╔====================================================================╗
2026-03-01 22:11:52,604 - INFO - ║               ESA COLLISION CHALLENGE SCORING                       ║
2026-03-01 22:11:52,607 - INFO - ╚====================================================================╝
2026-03-01 22:11:52,613 - INFO - 
2026-03-01 22:11:52,615 - INFO - ESA COLLISION AVOIDANCE CHALLENGE EVALUATION: XGBoost
2026-03-01 22:11:52,616 - INFO - ======================================================================
2026-03-01 22:11:52,622 - INFO - 
2026-03-01 22:11:52,624 - INFO - OFFICIAL ESA SCORE:
2026-03-01 22:11:52,625 - INFO -   Loss (L):           5.786930e+01  (LOWER IS BETTER)
2026-03-01 22:11:52,627 - INFO - 
2026-03-01 22:11:52,628 - INFO - COMPONENT METRICS:
2026-03-01 22:11:52,629 - INFO -   F2-Score:           0.252525  (max=1.0, higher is better)
2026-03-01 22:11:52,630 - INFO -   Precision:          0.679012  (TP / (T

## 18. F2-Optimized High-Risk Pipeline (Class Reweighting + SMOTE + Threshold Tuning + Ensemble)

This section implements the score-improvement items:
- Class reweighting
- Threshold tuning for recall/F2
- Additional risk-focused feature engineering
- Ensemble over multiple classifiers
- SMOTE oversampling for minority high-risk class
- Hyperparameter tuning with F2 as the optimization target


In [ ]:
# Optional installs for this section:
# !pip install -U imbalanced-learn xgboost lightgbm

from sklearn.metrics import fbeta_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def add_high_risk_indicator_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add compact, high-signal features for high-risk event detection.
    Works only on columns that already exist in the preprocessed event table.
    """
    out = df.copy()
    eps = 1e-9

    if {"relative_speed_mean", "miss_distance_mean"}.issubset(out.columns):
        out["speed_distance_ratio_v2"] = out["relative_speed_mean"] / (out["miss_distance_mean"].abs() + eps)

    if {"combined_cov_det", "miss_distance_mean"}.issubset(out.columns):
        out["covariance_to_distance"] = out["combined_cov_det"] / (out["miss_distance_mean"].abs() + eps)

    orbital_cols = [c for c in ["delta_sma", "delta_inc", "delta_ecc"] if c in out.columns]
    if orbital_cols:
        out["orbital_mismatch_l1"] = out[orbital_cols].abs().sum(axis=1)

    sigma_cols = [
        c for c in [
            "t_sigma_r_mean", "c_sigma_r_mean",
            "t_sigma_t_mean", "c_sigma_t_mean",
            "t_sigma_n_mean", "c_sigma_n_mean",
        ] if c in out.columns
    ]
    if sigma_cols:
        out["sigma_total"] = out[sigma_cols].sum(axis=1)

    if {"F10_mean", "AP_mean"}.issubset(out.columns):
        ssn = out["SSN_mean"] if "SSN_mean" in out.columns else 0
        out["space_weather_proxy"] = out["F10_mean"] + 0.4 * out["AP_mean"] + 0.05 * ssn

    interaction_pairs = [
        ("speed_to_distance", "combined_cov_det"),
        ("relative_speed_mean", "combined_cov_det"),
        ("miss_distance_mean", "combined_cov_det"),
    ]
    for a, b in interaction_pairs:
        if a in out.columns and b in out.columns:
            out[f"{a}_x_{b}"] = out[a] * out[b]

    return out

def tune_threshold_for_f2(y_true: np.ndarray, y_prob: np.ndarray, beta: float = 2.0) -> dict:
    """Tune classification threshold to maximize F-beta on validation set."""
    candidate_thresholds = np.linspace(0.05, 0.95, 181)
    best = {"threshold": 0.5, "f2": -1.0, "precision": 0.0, "recall": 0.0}
    for t in candidate_thresholds:
        y_pred = (y_prob >= t).astype(int)
        f2 = fbeta_score(y_true, y_pred, beta=beta, zero_division=0)
        if f2 > best["f2"]:
            best = {
                "threshold": float(t),
                "f2": float(f2),
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
            }
    return best

def evaluate_binary_predictions(y_true: np.ndarray, y_prob: np.ndarray, threshold: float, beta: float = 2.0) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    f2 = fbeta_score(y_true, y_pred, beta=beta, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "F2": float(f2),
        "Precision": float(precision),
        "Recall": float(recall),
        "TP": int(tp),
        "FP": int(fp),
        "FN": int(fn),
        "TN": int(tn),
        "Threshold": float(threshold),
    }

def prepare_high_risk_classification_data(
    df: pd.DataFrame,
    log_threshold: float = -6.0,
    test_size: float = 0.2,
    val_size: float = 0.2,
    random_state: int = 42,
) -> dict:
    """
    Split into train/val/test with stratification on high-risk class.
    Risk is expected in log10 scale; high-risk class is risk >= -6.
    """
    if "risk" not in df.columns:
        raise KeyError("Expected target column `risk` in dataframe.")

    y_binary = (pd.to_numeric(df["risk"], errors="coerce") >= log_threshold).astype(int)
    X = df.drop(columns=["risk", "event_id"], errors="ignore").copy()
    X = add_high_risk_indicator_features(X)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    logger.info(f"High-risk threshold (log10): {log_threshold}")
    logger.info(f"High-risk samples: {int(y_binary.sum())}/{len(y_binary)} ({100*y_binary.mean():.3f}%)")
    logger.info(f"Feature count after FE: {X.shape[1]}")

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y_binary,
        test_size=test_size,
        random_state=random_state,
        stratify=y_binary,
    )

    val_ratio_within_temp = val_size / (1.0 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp,
        test_size=val_ratio_within_temp,
        random_state=random_state,
        stratify=y_temp,
    )

    scaler = StandardScaler()
    X_train_s = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
    X_val_s = pd.DataFrame(scaler.transform(X_val), columns=X.columns, index=X_val.index)
    X_test_s = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

    return {
        "X_train": X_train_s,
        "X_val": X_val_s,
        "X_test": X_test_s,
        "y_train": y_train.values,
        "y_val": y_val.values,
        "y_test": y_test.values,
        "feature_names": list(X.columns),
    }

def maybe_apply_smote(X_train: pd.DataFrame, y_train: np.ndarray, random_state: int = 42):
    """Apply SMOTE only when imbalanced-learn is available and enough positives exist."""
    try:
        from imblearn.over_sampling import SMOTE
    except Exception:
        logger.warning("imbalanced-learn not installed. Continuing without SMOTE.")
        return X_train, y_train, False

    pos_count = int(np.sum(y_train == 1))
    if pos_count < 2:
        logger.warning("Not enough positive samples for SMOTE. Continuing without SMOTE.")
        return X_train, y_train, False

    k_neighbors = max(1, min(5, pos_count - 1))
    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    X_res, y_res = smote.fit_resample(X_train, y_train)
    X_res = pd.DataFrame(X_res, columns=X_train.columns)
    logger.info(f"SMOTE applied: {X_train.shape} -> {X_res.shape}")
    return X_res, y_res, True


In [31]:
def _build_candidate_models(scale_pos_weight: float, random_state: int = 42):
    """Create model + search-space tuples with class reweighting enabled."""
    candidates = []

    # 1) Logistic Regression
    logreg = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=random_state,
    )
    logreg_space = {"C": np.logspace(-2, 1, 12)}
    candidates.append(("LogisticRegression", logreg, logreg_space, 10))

    # 2) Random Forest
    rf = RandomForestClassifier(
        class_weight="balanced_subsample",
        random_state=random_state,
        n_jobs=-1,
    )
    rf_space = {
        "n_estimators": [200, 400, 700],
        "max_depth": [None, 8, 12, 18],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", 0.5],
    }
    candidates.append(("RandomForest", rf, rf_space, 16))

    # 3) Optional XGBoost
    try:
        import xgboost as xgb
        xgb_model = xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=random_state,
            n_jobs=-1,
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            verbosity=0,
        )
        xgb_space = {
            "n_estimators": [200, 400, 700],
            "max_depth": [3, 5, 7],
            "learning_rate": [0.02, 0.05, 0.1],
            "subsample": [0.7, 0.85, 1.0],
            "colsample_bytree": [0.7, 0.85, 1.0],
            "min_child_weight": [1, 3, 5],
            "gamma": [0, 0.5, 1.0],
        }
        candidates.append(("XGBoostClassifier", xgb_model, xgb_space, 20))
    except Exception:
        logger.warning("xgboost is not installed. Skipping XGBoostClassifier.")

    # 4) Optional LightGBM
    try:
        import lightgbm as lgb
        lgb_model = lgb.LGBMClassifier(
            objective="binary",
            class_weight={0: 1.0, 1: float(scale_pos_weight)},
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        lgb_space = {
            "n_estimators": [200, 400, 700],
            "learning_rate": [0.02, 0.05, 0.1],
            "max_depth": [-1, 6, 10],
            "num_leaves": [15, 31, 63],
            "subsample": [0.7, 0.85, 1.0],
            "colsample_bytree": [0.7, 0.85, 1.0],
            "min_child_samples": [20, 40, 80],
        }
        candidates.append(("LightGBMClassifier", lgb_model, lgb_space, 20))
    except Exception:
        logger.warning("lightgbm is not installed. Skipping LightGBMClassifier.")

    return candidates

def run_f2_optimized_high_risk_pipeline(df: pd.DataFrame, log_threshold: float = -6.0, random_state: int = 42):
    """
    End-to-end training/evaluation with:
    class reweighting + SMOTE + threshold tuning + ensemble + F2 hyperparameter search.
    """
    data_bundle = prepare_high_risk_classification_data(
        df=df,
        log_threshold=log_threshold,
        test_size=0.2,
        val_size=0.2,
        random_state=random_state,
    )

    X_train = data_bundle["X_train"]
    X_val = data_bundle["X_val"]
    X_test = data_bundle["X_test"]
    y_train = data_bundle["y_train"]
    y_val = data_bundle["y_val"]
    y_test = data_bundle["y_test"]

    pos = int(np.sum(y_train == 1))
    neg = int(np.sum(y_train == 0))
    scale_pos_weight = max(1.0, neg / max(pos, 1))
    logger.info(f"Class counts in train: pos={pos}, neg={neg}, scale_pos_weight={scale_pos_weight:.2f}")

    X_train_bal, y_train_bal, smote_used = maybe_apply_smote(X_train, y_train, random_state=random_state)

    f2_scorer = make_scorer(fbeta_score, beta=2, zero_division=0)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=random_state)
    candidates = _build_candidate_models(scale_pos_weight=scale_pos_weight, random_state=random_state)

    model_records = []
    trained_models = {}
    val_prob_store = {}

    for model_name, estimator, param_space, n_iter in candidates:
        logger.info(f"Training {model_name} with RandomizedSearchCV (n_iter={n_iter})")
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=param_space,
            n_iter=n_iter,
            scoring=f2_scorer,
            n_jobs=-1,
            cv=cv,
            verbose=0,
            random_state=random_state,
            refit=True,
        )
        search.fit(X_train_bal, y_train_bal)

        best_model = search.best_estimator_
        best_cv_f2 = float(search.best_score_)

        val_prob = best_model.predict_proba(X_val)[:, 1]
        threshold_info = tune_threshold_for_f2(y_val, val_prob, beta=2.0)
        val_metrics = evaluate_binary_predictions(y_val, val_prob, threshold=threshold_info["threshold"], beta=2.0)

        test_prob = best_model.predict_proba(X_test)[:, 1]
        test_metrics = evaluate_binary_predictions(y_test, test_prob, threshold=threshold_info["threshold"], beta=2.0)

        logger.info(
            f"{model_name} | CV F2={best_cv_f2:.4f} | "
            f"Val F2={val_metrics['F2']:.4f} | Test F2={test_metrics['F2']:.4f} | "
            f"Threshold={threshold_info['threshold']:.3f}"
        )

        trained_models[model_name] = {
            "model": best_model,
            "threshold": float(threshold_info["threshold"]),
            "best_cv_f2": best_cv_f2,
        }
        val_prob_store[model_name] = val_prob

        model_records.append({
            "Model": model_name,
            "Split": "validation",
            "Threshold": float(threshold_info["threshold"]),
            "CV_Best_F2": best_cv_f2,
            **val_metrics,
            "SMOTE_Used": bool(smote_used),
        })
        model_records.append({
            "Model": model_name,
            "Split": "test",
            "Threshold": float(threshold_info["threshold"]),
            "CV_Best_F2": best_cv_f2,
            **test_metrics,
            "SMOTE_Used": bool(smote_used),
        })

    # Simple soft-voting ensemble over available models
    if val_prob_store:
        val_prob_matrix = np.column_stack([val_prob_store[name] for name in val_prob_store])
        val_prob_ens = val_prob_matrix.mean(axis=1)
        ens_threshold = tune_threshold_for_f2(y_val, val_prob_ens, beta=2.0)["threshold"]
        val_metrics_ens = evaluate_binary_predictions(y_val, val_prob_ens, threshold=ens_threshold, beta=2.0)

        test_prob_matrix = np.column_stack([trained_models[name]["model"].predict_proba(X_test)[:, 1] for name in val_prob_store])
        test_prob_ens = test_prob_matrix.mean(axis=1)
        test_metrics_ens = evaluate_binary_predictions(y_test, test_prob_ens, threshold=ens_threshold, beta=2.0)

        model_records.append({
            "Model": "EnsembleSoftVote",
            "Split": "validation",
            "Threshold": float(ens_threshold),
            "CV_Best_F2": np.nan,
            **val_metrics_ens,
            "SMOTE_Used": bool(smote_used),
        })
        model_records.append({
            "Model": "EnsembleSoftVote",
            "Split": "test",
            "Threshold": float(ens_threshold),
            "CV_Best_F2": np.nan,
            **test_metrics_ens,
            "SMOTE_Used": bool(smote_used),
        })

    results_df = pd.DataFrame(model_records)
    results_df = results_df.sort_values(["Split", "F2"], ascending=[True, False]).reset_index(drop=True)

    output_dir = "outputs"
    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, "high_risk_classification_results.csv")
    results_df.to_csv(out_path, index=False)
    logger.info(f"Saved classification comparison to: {out_path}")

    return results_df, trained_models


In [ ]:
# Run high-risk optimization pipeline
base_df_for_cls = preprocessed_df.copy() if "preprocessed_df" in globals() else get_preprocessed_event_table()

cls_results_df, cls_models = run_f2_optimized_high_risk_pipeline(
    df=base_df_for_cls,
    log_threshold=-6.0,
    random_state=42,
)

display_cols = ["Model", "Split", "F2", "Precision", "Recall", "Threshold", "TP", "FP", "FN", "TN", "SMOTE_Used"]
cls_results_df[display_cols]


In [36]:
# Test cell to read the output of improved pipeline

# Check if models_improved_df exists from the previous cell execution
try:
    print("Improved Pipeline Results:")
    print(results_improved_df)
except NameError:
    print("Cell not executed yet - running quick test of ESA scoring...")
    
# Quick display of improved scores if available
try:
    results_csv = Path("outputs/esa_scores_improved.csv")
    if results_csv.exists():
        improved_results = pd.read_csv(results_csv)
        print("\nImproved Pipeline Scores:")
        print(improved_results[['Model', 'ESA_Loss', 'F2_Score', 'Recall', 'Precision']])
except:
    pass

Improved Pipeline Results:
Cell not executed yet - running quick test of ESA scoring...
